# TAIKO® Grinding Process Optimisation

**DISCO TAIKO®** — ultra-thin wafer process for HBM packaging.
Leaves a thick edge ring (~3 mm) for mechanical support while grinding the
center to ≤ 25 µm (HBM4).

**This notebook implements the グラインディング研究方針:**
- **案A**: Residual stress → wafer warpage prediction (GP surrogate)
- **案C**: Bayesian optimisation of grinding recipe (warpage + MRR)

Reference: Wu et al. (2023) *Journal of Mechanics* 39:191–198

In [ ]:
import sys, os

NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() \
               else os.path.abspath(os.path.join(os.getcwd(),
                    '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
REPO_ROOT = os.path.dirname(NOTEBOOK_DIR) if os.path.basename(NOTEBOOK_DIR) == 'notebooks' \
            else NOTEBOOK_DIR
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from taiko.optimizer import TAIKOOptimizer, optimize_taiko_recipe, kabra_stress_estimate

print(f'REPO_ROOT: {REPO_ROOT}')
print('TAIKOOptimizer loaded')

## 1. Dataset — Wu et al. (2023) Taguchi L9

9 grinding experiments on 300 mm Si wafer, 2-stage grinding.

In [2]:
# Taguchi L9: [z2_wheel_speed_rpm, z2_feed_um_s, z1_wafer_speed_rpm] → warpage_mm
X_raw = np.array([
    [1400, 0.25, 150], [1400, 0.30, 200], [1400, 0.35, 250],
    [1800, 0.25, 200], [1800, 0.30, 250], [1800, 0.35, 150],
    [2200, 0.25, 250], [2200, 0.30, 150], [2200, 0.35, 200],
], dtype=float)

y_raw = np.array([0.082, 0.091, 0.098, 0.063, 0.071, 0.079,
                   0.045, 0.053, 0.061])

feat_names = ['Z2 wheel speed [rpm]', 'Z2 feed rate [µm/s]', 'Z1 wafer speed [rpm]']
print(f'Dataset: {len(X_raw)} experiments, warpage range {y_raw.min()*1e3:.0f}–{y_raw.max()*1e3:.0f} µm')

Dataset: 9 experiments, warpage range 45–98 µm


## 2. GP Surrogate — Leave-One-Out Cross-Validation

In [3]:
opt = TAIKOOptimizer()
opt.fit_from_data(X_raw, y_raw)

# LOO-CV
loo_pred = []
for i in range(len(X_raw)):
    X_tr = np.delete(X_raw, i, axis=0)
    y_tr = np.delete(y_raw, i)
    opt_loo = TAIKOOptimizer()
    opt_loo.fit_from_data(X_tr, y_tr)
    mu, _ = opt_loo.predict(X_raw[[i]])
    loo_pred.append(mu[0])

loo_pred = np.array(loo_pred)
rmse = np.sqrt(np.mean((loo_pred - y_raw)**2))
r2   = 1 - np.sum((loo_pred - y_raw)**2) / np.sum((y_raw - y_raw.mean())**2)
print(f'LOO-CV  RMSE = {rmse*1e3:.2f} µm   R² = {r2:.3f}')

LOO-CV  RMSE = 1.12 µm   R² = 0.995


## 3. GP Response Surface

In [ ]:
# 2-D slice: Z2 wheel speed × Z2 feed rate, Z1 wafer speed fixed at 200 rpm
ws_grid   = np.linspace(1400, 2200, 60)
feed_grid = np.linspace(0.25, 0.35, 60)
WS, FEED  = np.meshgrid(ws_grid, feed_grid)

X_grid = np.column_stack([WS.ravel(), FEED.ravel(), np.full(WS.size, 200.0)])
mu_grid, std_grid = opt.predict(X_grid)
MU  = mu_grid.reshape(WS.shape) * 1e3   # → µm
STD = std_grid.reshape(WS.shape) * 1e3

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

c1 = axes[0].contourf(WS, FEED, MU, levels=20, cmap='RdYlGn_r')
axes[0].scatter(X_raw[:,0], X_raw[:,1], c=y_raw*1e3, s=80, edgecolors='k',
                cmap='RdYlGn_r', zorder=5, label='Taguchi data')
plt.colorbar(c1, ax=axes[0], label='Warpage [µm]')
axes[0].set_xlabel('Z2 Wheel Speed [rpm]')
axes[0].set_ylabel('Z2 Feed Rate [µm/s]')
axes[0].set_title('GP Predicted Warpage (Z1=200 rpm)')
axes[0].legend()

c2 = axes[1].contourf(WS, FEED, STD, levels=20, cmap='Blues')
plt.colorbar(c2, ax=axes[1], label='Std [µm]')
axes[1].set_xlabel('Z2 Wheel Speed [rpm]')
axes[1].set_ylabel('Z2 Feed Rate [µm/s]')
axes[1].set_title('GP Uncertainty (exploration target)')

plt.suptitle('TAIKO® Grinding — GP Response Surface', fontweight='bold')
plt.tight_layout()
out_path = os.path.join(REPO_ROOT, 'notebooks', 'taiko_gp_surface.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 4. Bayesian Optimisation — Recipe Search

In [5]:
opt_bo = TAIKOOptimizer(w_warpage=1.0, w_kabra=0.3)
opt_bo.fit_from_data(X_raw, y_raw)

best = opt_bo.optimize(n_iter=25)

print('\nOptimal TAIKO® recipe:')
print(f'  Z2 wheel speed : {best["z2_wheel_speed_rpm"]:.0f} rpm')
print(f'  Z2 feed rate   : {best["z2_feed_um_s"]:.3f} µm/s')
print(f'  Z1 wafer speed : {best["z1_wafer_speed_rpm"]:.0f} rpm')
print(f'  Predicted warpage: {best["warpage_mm"]*1e3:.1f} µm')


Optimal TAIKO® recipe:
  Z2 wheel speed : 2200 rpm
  Z2 feed rate   : 0.250 µm/s
  Z1 wafer speed : 150 rpm
  Predicted warpage: 44.1 µm


## 5. BO Convergence + Pareto Front

In [ ]:
# BO convergence trace
y_trace = opt_bo._y_train * 1e3  # µm
best_so_far = np.minimum.accumulate(y_trace)

# Pareto front: warpage vs KABRA stress
front = opt_bo.pareto_front(n_grid=20)
front_warpage = np.array([p['warpage_mm'] * 1e3 for p in front])
front_kabra   = np.array([p['kabra_stress_score'] for p in front])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Convergence
axes[0].plot(y_trace, 'o-', ms=3, lw=1, alpha=0.5, color='steelblue', label='Observations')
axes[0].plot(best_so_far, lw=2, color='tomato', label='Best so far')
axes[0].axvline(len(X_raw)-1, color='gray', ls='--', lw=1, label='Taguchi data ends')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Warpage [µm]')
axes[0].set_title('BO Convergence')
axes[0].legend()

# Pareto front
idx = np.argsort(front_warpage)
axes[1].plot(front_warpage[idx], front_kabra[idx], 'o-', ms=5, color='mediumseagreen',
             label='Pareto front')
axes[1].scatter([best['warpage_mm']*1e3], [best['kabra_stress_score']],
                s=120, zorder=5, color='red', label='BO optimum')
axes[1].set_xlabel('Warpage [µm]')
axes[1].set_ylabel('KABRA stress score')
axes[1].set_title('Pareto Front: Warpage vs. KABRA Stress')
axes[1].legend()

plt.suptitle('TAIKO® BO — Convergence & Pareto', fontweight='bold')
plt.tight_layout()
out_path = os.path.join(REPO_ROOT, 'notebooks', 'taiko_bo_result.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 6. Summary — HBM4 Feasibility

HBM4 specification: center thickness ≤ 25 µm, warpage ≤ 100 µm.

In [7]:
from taiko.optimizer import taiko_edge_feasibility

print('=== HBM4 TAIKO® Feasibility Check ===')
for edge_w in [2.0, 3.0, 4.0]:
    for thick in [25.0, 50.0]:
        ok = taiko_edge_feasibility(edge_w, thick)
        best_run = optimize_taiko_recipe(
            n_iter=10, edge_width_mm=edge_w, thin_thickness_um=thick, verbose=False
        ) if ok else None
        warp = f'{best_run["warpage_mm"]*1e3:.1f} µm' if ok else 'INFEASIBLE'
        hbm4 = '✓ HBM4 OK' if (ok and best_run['warpage_mm']*1e3 <= 100) else ('✗' if ok else '—')
        print(f'  edge={edge_w:.0f}mm / thin={thick:.0f}µm → warpage={warp:12s}  {hbm4}')

=== HBM4 TAIKO® Feasibility Check ===


  edge=2mm / thin=25µm → warpage=42.4 µm       ✓ HBM4 OK


  edge=2mm / thin=50µm → warpage=42.1 µm       ✓ HBM4 OK


  edge=3mm / thin=25µm → warpage=42.0 µm       ✓ HBM4 OK


  edge=3mm / thin=50µm → warpage=43.6 µm       ✓ HBM4 OK


  edge=4mm / thin=25µm → warpage=41.6 µm       ✓ HBM4 OK


  edge=4mm / thin=50µm → warpage=40.3 µm       ✓ HBM4 OK
